In [1]:
import os
import glob
import numpy as np
import pandas as pd

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.common.qc import run_session_synapse_qc
from vip_slap2_analysis.behavior.preprocess import process_behavior_session
from vip_slap2_analysis.glutamate.extraction import process_glutamate_extraction
from vip_slap2_analysis.calcium.qc import run_calcium_qc
from vip_slap2_analysis.calcium.extraction import process_calcium_extraction

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

### Decla

In [3]:
target_mice = [
    803496,
#     804730,804733,810196,
#     809047,803121,
#     826033,838410,834788
]

### Load session registry

In [4]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check","volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [5]:
meta = {
    "epochs_mode": "continuous",
    "prepost_sec": {
        "image_identity": (0.25, 0.50),
        "change": (1.00, 0.75),
        "omission": (1.00, 1.50),
    },
}

### Process all data and sessions

In [6]:
for asset in assets:
    print(f'<<<PROCESSING: SUBJECT: {asset.subject_id}, SESSION: {asset.session_id}>>>')
    print(f'Initiating QC and processing for {asset.session_id}')
    print("summary:", asset.summary_mat)
    print("bonsai:", asset.bonsai_event_log_csv)
    print("harp:", asset.harp_dir)
    print("photodiode:", asset.photodiode_pkl)
    print('')
    asset.ensure_dirs()
    
    print(f"Processing behavior data...")
    
    behavior_result = process_behavior_session(
        asset,
        metadata=meta,
        overwrite_harp_extract=False,
        overwrite_alignment=False,
        save_corrected_in_place=True,
        min_epoch_duration=6.0,
        trial_gap_start=0.02,
        expected_trial_epoch_min=None,
        use_piecewise_warp=True,
    )

    print('Result status: ',behavior_result.status)
    print('Failures: ',behavior_result.failure_stage)
    print('Failure cause: ',behavior_result.failure_reasons)

    if not behavior_result.ready_for_physiology_extraction:
        print(f"Skipping physiology extraction for {asset.session_id}")
        print('\n')
        continue
    
    print(f"\nProcessing physiology data...")
    physiology_result = run_session_synapse_qc(
        summary_mat_path=asset.summary_mat,
        output_dir=asset.qc_dir/"slap2",
        session_id=asset.session_id,
        subject_id=asset.subject_id,
        trace_signal='dF',
        trace_mode='ls',
        save=True,
        make_plots=True)
    print(f'Finished running SLAP2 glutamate QC on {asset.session_id}\n')
    
    print('Extracting physiology data...')
    glu_result = process_glutamate_extraction(
    asset,
    metadata={
        "im_rate_hz": 200.0,
        "prepost_sec": {
            "image": (0.25, 0.50),
            "change": (1.00, 0.75),
            "omission": (1.00, 1.50),
        }
    },
    use_synapse_qc=False,
    overwrite=True,
    )
    print(f'Finished extracting glutamate data for {asset.session_id}\n')
    
    print(f'Checking Ca2+ data...')
    indicator2 = None
    if getattr(asset, "metadata", None):
        indicator2 = asset.metadata.get("indicator2")

    has_calcium = isinstance(indicator2, str) and ("RCaMP" in indicator2 or "jRGECO" in indicator2 or "CaMP" in indicator2)

    if has_calcium:
        print("Running QC on Ca2+ data...")
        ca_qc = run_calcium_qc(
            asset,
            metadata={"im_rate_hz": 200.0},
            overwrite=True,
            motion_correct=True,
            max_session_minutes=None,
        )

        print("Extracting Ca2+ data...")
        ca_result = process_calcium_extraction(
            asset,
            metadata={
                "im_rate_hz": 200.0,
                "prepost_sec": {
                    "image": (0.25, 0.50),
                    "change": (1.00, 0.75),
                    "omission": (1.00, 1.50),
                },
            },
            use_soma_qc=True,
            motion_correct=True,
            max_session_minutes=None,
            overwrite=True,
        )
        print(f'Finished extracting Ca2+ data for {asset.session_id}')
    else:
        print(f"Skipping Ca2+ QC/extraction for {asset.session_id}: no indicator2 / no Ca channel")
    print('')

<<<PROCESSING: SUBJECT: 803496, SESSION: 803496_2025-07-25_13-02-10>>>
Initiating QC and processing for 803496_2025-07-25_13-02-10
summary: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10_slap2_2026-01-18_05-53-08\source_extraction\ExperimentSummary\SummaryLoCo-260117-185357.mat
bonsai: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10\behavior\VCO1_Behavior.harp\bonsai_event_log.csv
harp: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10\behavior\VCO1_Behavior.harp
photodiode: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10\behavior\VCO1_Behavior.harp\extracted_files\photodiode.pkl


Processing physiology data...
Finished running SLAP2 glutamate QC on 803496_2025-07-25_13-02-10

Extracting physiology data...
F

C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis\glutamate\alignment.py:511: RuntimeWarning: Mean of empty slice
  "mean": np.nanmean(stack, axis=0),
C:\Users\andrew.shelton\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Finished extracting glutamate data for 803496_2025-07-29_13-34-35

<<<PROCESSING: SUBJECT: 803496, SESSION: 803496_2025-07-30_10-05-23>>>
Initiating QC and processing for 803496_2025-07-30_10-05-23
summary: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-30_803496\803496_2025-07-30_10-05-23_slap2_2026-01-20_19-56-29\source_extraction\ExperimentSummary\SummaryLoCo-260120-091441.mat
bonsai: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-30_803496\803496_2025-07-30_10-05-23\behavior\VCO1_Behavior.harp\bonsai_event_log.csv
harp: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-30_803496\803496_2025-07-30_10-05-23\behavior\VCO1_Behavior.harp
photodiode: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-30_803496\803496_2025-07-30_10-05-23\behavior\VCO1_Behavior.harp\extracted_files\photodiode.pkl


Processing physiology data...
Finished running SLAP2 glutamat

C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis\glutamate\alignment.py:511: RuntimeWarning: Mean of empty slice
  "mean": np.nanmean(stack, axis=0),
C:\Users\andrew.shelton\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Finished extracting glutamate data for 803496_2025-07-31_09-43-28

<<<PROCESSING: SUBJECT: 803496, SESSION: 803496_2025-08-01_13-22-49>>>
Initiating QC and processing for 803496_2025-08-01_13-22-49
summary: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-08-01_803496\803496_2025-08-01_13-22-49_slap2_2026-01-21_19-49-17\source_extraction\ExperimentSummary\SummaryLoCo-260121-103948.mat
bonsai: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-08-01_803496\803496_2025-08-01_13-22-49\behavior\VCO1_Behavior.harp\bonsai_event_log.csv
harp: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-08-01_803496\803496_2025-08-01_13-22-49\behavior\VCO1_Behavior.harp
photodiode: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-08-01_803496\803496_2025-08-01_13-22-49\behavior\VCO1_Behavior.harp\extracted_files\photodiode.pkl


Processing physiology data...
Finished running SLAP2 glutamat

...

Finished extracting glutamate data for 838410_2026-03-05_10-16-37

<<<PROCESSING: SUBJECT: 838410, SESSION: 838410_2026-03-18_16-43-23>>>
Initiating QC and processing for 838410_2026-03-18_16-43-23
summary: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-18_16-43-23\838410_2026-03-18_16-43-23_slap2_2026-03-18_16-43-23\source_extraction\ExperimentSummary\SummaryLoCo-260320-073439.mat
bonsai: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-18_16-43-23\838410_2026-03-18_16-43-23\behavior\VCO1_Behavior.harp\bonsai_event_log.csv
harp: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-18_16-43-23\838410_2026-03-18_16-43-23\behavior\VCO1_Behavior.harp
photodiode: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-18_16-43-23\838410_2026-03-18_16-43-23\behavior\VCO1_Behavior.harp\extracted_files\photodiode.pkl

C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis\glutamate\alignment.py:511: RuntimeWarning: Mean of empty slice
  "mean": np.nanmean(stack, axis=0),
C:\Users\andrew.shelton\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Finished extracting glutamate data for 838410_2026-03-18_16-43-23

<<<PROCESSING: SUBJECT: 838410, SESSION: 838410_2026-03-19_13-06-48>>>
Initiating QC and processing for 838410_2026-03-19_13-06-48
summary: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-19_13-06-48\838410_2026-03-19_13-06-48_slap2_2026-03-19_13-06-48\source_extraction\ExperimentSummary\SummaryLoCo-260322-111723.mat
bonsai: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-19_13-06-48\838410_2026-03-19_13-06-48\behavior\VCO1_Behavior.harp\bonsai_event_log.csv
harp: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-19_13-06-48\838410_2026-03-19_13-06-48\behavior\VCO1_Behavior.harp
photodiode: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-19_13-06-48\838410_2026-03-19_13-06-48\behavior\VCO1_Behavior.harp\extracted_files\photodiode.pkl

C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis\glutamate\alignment.py:511: RuntimeWarning: Mean of empty slice
  "mean": np.nanmean(stack, axis=0),
C:\Users\andrew.shelton\Anaconda3\lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Finished extracting glutamate data for 838410_2026-03-19_13-06-48

<<<PROCESSING: SUBJECT: 838410, SESSION: 838410_2026-03-20_10-00-59>>>
Initiating QC and processing for 838410_2026-03-20_10-00-59
summary: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-20_10-00-59\838410_2026-03-20_10-00-59_slap2_2026-03-20_10-00-59\source_extraction\ExperimentSummary\SummaryLoCo-260323-174847.mat
bonsai: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-20_10-00-59\838410_2026-03-20_10-00-59\behavior\VCO1_Behavior.harp\bonsai_event_log.csv
harp: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-20_10-00-59\838410_2026-03-20_10-00-59\behavior\VCO1_Behavior.harp
photodiode: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-20_10-00-59\838410_2026-03-20_10-00-59\behavior\VCO1_Behavior.harp\extracted_files\photodiode.pkl

###